# BƯỚC 1: Nạp dữ liệu và kiểm tra ban đầu (Data Loading & Initial Exploration)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Cấu hình hiển thị pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

In [2]:
# Đường dẫn tới file dữ liệu gốc
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/accepted_2007_to_2018Q4.csv")

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

In [3]:
n_rows, n_cols = df_raw.shape
mem_gb = df_raw.memory_usage(deep=True).sum() / 1e9

print(f"Tổng số dòng: {n_rows:,}")
print(f"Tổng số cột: {n_cols}")
print(f"Dung lượng DataFrame chiếm dụng trong RAM: {mem_gb:.2f} GB")

Tổng số dòng: 2,260,701
Tổng số cột: 151
Dung lượng DataFrame chiếm dụng trong RAM: 6.28 GB


In [4]:
def summarize_columns(df):
    first_valid = df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None)
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100),
        "nunique": df.nunique(),
        "sample_val": first_valid
    })
    return summary

In [5]:
col_summary = summarize_columns(df_raw)

print("Phân bố số lượng cột theo kiểu dữ liệu:")
print(col_summary["dtype"].value_counts())

Phân bố số lượng cột theo kiểu dữ liệu:
dtype
float64    113
str         38
Name: count, dtype: int64


In [6]:
heavy_missing = col_summary[col_summary["missing_pct"] > 80]
print(f"Số cột bị khuyết trên 80% dữ liệu: {len(heavy_missing)} / {n_cols} cột")
print("Top 10 cột có tỷ lệ missing cao nhất:")
heavy_missing.sort_values(by="missing_pct", ascending=False)[["missing_pct", "nunique", "dtype"]].head(10)

Số cột bị khuyết trên 80% dữ liệu: 39 / 151 cột
Top 10 cột có tỷ lệ missing cao nhất:


,missing_pct,nunique,dtype
member_id,100.000,0,float64
orig_projected_additional_accrued_interest,99.617,7487,float64
hardship_end_date,99.517,28,str
hardship_reason,99.517,9,str
hardship_status,99.517,3,str
hardship_type,99.517,1,str
hardship_start_date,99.517,27,str
hardship_loan_status,99.517,5,str
hardship_dpd,99.517,34,float64
hardship_length,99.517,1,float64


In [7]:
print("Phân bố các giá trị trong cột loan_status:")
print(df_raw["loan_status"].value_counts(dropna=False))

Phân bố các giá trị trong cột loan_status:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64


# BƯỚC 2: EDA thời gian & Chia tập Train / Val / Test (70/15/15) theo Time-Series

In [8]:
# Lọc 3 trạng thái: Trả hết (0) và Vỡ nợ (Charged Off + Default -> 1)
valid_status = ["Fully Paid", "Charged Off", "Default"]
df_labeled = df_raw[df_raw["loan_status"].isin(valid_status)].copy()

# Gán nhãn target: Charged Off hoặc Default đều là vỡ nợ (1)
df_labeled["target"] = df_labeled["loan_status"].isin(["Charged Off", "Default"]).astype(int)

In [9]:
status_breakdown = df_labeled["loan_status"].value_counts()
print("Số lượng từng trạng thái sau khi lọc:")
print(status_breakdown)

n_default_status = (df_labeled["loan_status"] == "Default").sum()
print(f"\n-> Nhóm Default đóng góp thêm {n_default_status} ca vỡ nợ vào tập dữ liệu.")
print(f"-> Tổng số ca vỡ nợ (target=1): {df_labeled['target'].sum():,}")
print(f"-> Tỷ lệ vỡ nợ (Default Rate) toàn bộ: {df_labeled['target'].mean()*100:.2f}%")

Số lượng từng trạng thái sau khi lọc:
loan_status
Fully Paid     1076751
Charged Off     268559
Default             40
Name: count, dtype: int64

-> Nhóm Default đóng góp thêm 40 ca vỡ nợ vào tập dữ liệu.
-> Tổng số ca vỡ nợ (target=1): 268,599
-> Tỷ lệ vỡ nợ (Default Rate) toàn bộ: 19.96%


In [10]:
# Chuyển đổi issue_d sang datetime
df_labeled["issue_date"] = pd.to_datetime(df_labeled["issue_d"], format="%b-%Y")

# In khoảng thời gian dữ liệu bao phủ
min_date = df_labeled["issue_date"].min().strftime("%m/%Y")
max_date = df_labeled["issue_date"].max().strftime("%m/%Y")
print(f"Dữ liệu trải dài từ tháng: {min_date} đến tháng: {max_date}")

Dữ liệu trải dài từ tháng: 06/2007 đến tháng: 12/2018


In [11]:
df_labeled["issue_year"] = df_labeled["issue_date"].dt.year
yearly_stats = df_labeled.groupby("issue_year").agg(
    total_loans=("target", "count"),
    default_rate=("target", lambda x: x.mean() * 100)
)
print("Thống kê khoản vay theo từng năm phát hành:")
print(yearly_stats)

Thống kê khoản vay theo từng năm phát hành:
            total_loans  default_rate
issue_year                           
2007                251        17.928
2008               1562        15.813
2009               4716        12.595
2010              11536        12.890
2011              21721        15.179
2012              53367        16.197
2013             134804        15.596
2014             223103        18.450
2015             375546        20.185
2016             293105        23.286
2017             169321        23.133
2018              56318        15.757


In [12]:
# Sắp xếp theo thời gian tăng dần
df_sorted = df_labeled.sort_values(by=["issue_date", "id"]).reset_index(drop=True)

# Xác định vị trí cắt dòng 70% / 15% / 15%
idx_train_end = int(len(df_sorted) * 0.70)
idx_val_end = int(len(df_sorted) * 0.85)

print(f"Chỉ số cắt tập Train: từ dòng 0 đến {idx_train_end:,}")
print(f"Chỉ số cắt tập Val:   từ dòng {idx_train_end:,} đến {idx_val_end:,}")
print(f"Chỉ số cắt tập Test:  từ dòng {idx_val_end:,} đến {len(df_sorted):,}")

Chỉ số cắt tập Train: từ dòng 0 đến 941,744
Chỉ số cắt tập Val:   từ dòng 941,744 đến 1,143,547
Chỉ số cắt tập Test:  từ dòng 1,143,547 đến 1,345,350


In [13]:
train_df = df_sorted.iloc[:idx_train_end].copy()
val_df = df_sorted.iloc[idx_train_end:idx_val_end].copy()
test_df = df_sorted.iloc[idx_val_end:].copy()

print(f"Train set: {len(train_df):,} dòng | Từ {train_df['issue_date'].min():%Y-%m} đến {train_df['issue_date'].max():%Y-%m}")
print(f"Val set:   {len(val_df):,} dòng | Từ {val_df['issue_date'].min():%Y-%m} đến {val_df['issue_date'].max():%Y-%m}")
print(f"Test set:  {len(test_df):,} dòng | Từ {test_df['issue_date'].min():%Y-%m} đến {test_df['issue_date'].max():%Y-%m}")

Train set: 941,744 dòng | Từ 2007-06 đến 2016-04
Val set:   201,803 dòng | Từ 2016-04 đến 2017-02
Test set:  201,803 dòng | Từ 2017-02 đến 2018-12


In [14]:
# Kiểm tra logic thời gian
assert train_df["issue_date"].max() <= val_df["issue_date"].min(), "Lỗi: Train bị chồng lấn thời gian với Val!"
assert val_df["issue_date"].max() <= test_df["issue_date"].min(), "Lỗi: Val bị chồng lấn thời gian với Test!"

print(" Xác nhận kiểm tra logic thành công: Thứ tự thời gian Train <= Val <= Test được đảm bảo tuyệt đối!")

 Xác nhận kiểm tra logic thành công: Thứ tự thời gian Train <= Val <= Test được đảm bảo tuyệt đối!


In [15]:
print(f"Tỷ lệ vỡ nợ tập Train: {train_df['target'].mean()*100:.2f}%")
print(f"Tỷ lệ vỡ nợ tập Val:   {val_df['target'].mean()*100:.2f}%")
print(f"Tỷ lệ vỡ nợ tập Test:  {test_df['target'].mean()*100:.2f}%")

Tỷ lệ vỡ nợ tập Train: 18.69%
Tỷ lệ vỡ nợ tập Val:   24.79%
Tỷ lệ vỡ nợ tập Test:  21.08%


# BƯỚC 3: Sàng lọc 27 Feature hợp lệ tại thời điểm nộp đơn (Application-Time Whitelist)

## 3.1 Phễu lọc từ 151 cột xuống 27 feature

Thay vì gõ sẵn danh sách 27 cột, ta đi từ 151 cột của file gốc và loại dần theo 5 quy tắc, mỗi quy tắc một hàm/ô code riêng. Mọi cột bị loại ở mỗi bước đều được in ra số lượng, và cuối cùng danh sách còn lại phải **trùng khớp** với whitelist 27 cột ở mục 3.2.

| Bước | Quy tắc | Nguồn quyết định |
|---|---|---|
| L1 | Loại cột không biết được lúc nộp đơn (định danh, thanh toán sau khi vay, hỗ trợ khó khăn, tất toán nợ) | Từ điển ở notebook 1, mục 1.3 (là giả thuyết, chưa tự kiểm chứng ở notebook này) |
| L2 | Loại cột văn bản tự do / định danh (`title`, `desc`, `emp_title`, `zip_code`) | Quyết định: nhiều giá trị khác nhau, không dùng trực tiếp được trong mô hình tuyến tính |
| L3 | Loại cột thiếu quá `MAX_MISSING`% trên **train** | Tính từ dữ liệu train |
| L4 | Loại cột hoàn toàn vắng mặt ở giai đoạn đầu của train (trước `EARLY_CUTOFF`) | Tính từ dữ liệu train |
| L5 | Loại cột số gần như hằng số (một giá trị chiếm từ `NEAR_CONSTANT`% trở lên trên train) | Tính từ dữ liệu train |

Không có ngoại lệ: mọi cột bị loại hay được giữ đều do đúng năm quy tắc trên.

In [16]:
COT_KHONG_BIET_LUC_NOP_DON = [
    # initial_list_status (từ điển ghi "Không rõ") cũng bị loại
    "initial_list_status",
    # Thông tin khoản vay (11 cột)
    "id", "member_id", "url", "funded_amnt", "funded_amnt_inv", "issue_d", "loan_status",
    "pymnt_plan", "policy_code", "disbursement_method",
    # Thanh toán sau khi vay (15 cột)
    "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv", "total_rec_prncp",
    "total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee", "last_pymnt_d",
    "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d", "last_fico_range_high",
    "last_fico_range_low",
    # Hỗ trợ khó khăn tài chính (15 cột)
    "hardship_flag", "hardship_type", "hardship_reason", "hardship_status", "deferral_term",
    "hardship_amount", "hardship_start_date", "hardship_end_date", "payment_plan_start_date",
    "hardship_length", "hardship_dpd", "hardship_loan_status",
    "orig_projected_additional_accrued_interest", "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",
    # Tất toán nợ (7 cột)
    "debt_settlement_flag", "debt_settlement_flag_date", "settlement_status", "settlement_date",
    "settlement_amount", "settlement_percentage", "settlement_term",
]

In [17]:
# Các tham số của phễu lọc (mọi ngưỡng đều chỉ được áp dụng với số liệu tính trên train)
COT_VAN_BAN_DINH_DANH = ["title", "desc", "emp_title", "zip_code"]
MAX_MISSING = 60        # % thiếu tối đa trên train
EARLY_CUTOFF = pd.Timestamp("2012-01-01")
NEAR_CONSTANT = 98      # % tối thiểu của giá trị phổ biến nhất để coi là gần hằng số

In [18]:
def drop_listed(columns, to_drop):
    return [c for c in columns if c not in to_drop]

In [19]:
def columns_too_missing(df, columns, max_missing):
    missing_pct = df[columns].isna().mean() * 100
    return missing_pct[missing_pct > max_missing].index.tolist()

In [20]:
def columns_absent_early(df, columns, cutoff):
    early = df[df["issue_date"] < cutoff]
    return [c for c in columns if early[c].isna().all()]

In [21]:
def columns_near_constant(df, columns, threshold):
    result = []
    for c in columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        top_share = df[c].value_counts(normalize=True, dropna=True).iloc[0] * 100
        if top_share >= threshold:
            result.append(c)
    return result

In [22]:
cols_start = list(df_raw.columns)
funnel = [("Ban đầu", "151 cột của file gốc", [])]

# L1: không biết được lúc nộp đơn
dropped = [c for c in cols_start if c in COT_KHONG_BIET_LUC_NOP_DON]
cols_l1 = drop_listed(cols_start, dropped)
funnel.append(("L1", "Không biết lúc nộp đơn (theo từ điển)", dropped))

# L2: văn bản tự do / định danh
dropped = [c for c in cols_l1 if c in COT_VAN_BAN_DINH_DANH]
cols_l2 = drop_listed(cols_l1, dropped)
funnel.append(("L2", "Văn bản tự do / định danh", dropped))

# L3: thiếu quá nhiều trên train
dropped = columns_too_missing(train_df, cols_l2, MAX_MISSING)
cols_l3 = drop_listed(cols_l2, dropped)
funnel.append(("L3", f"Thiếu trên {MAX_MISSING}% ở train", dropped))

# L4: vắng mặt hoàn toàn ở giai đoạn đầu của train (trừ cột giữ theo quyết định)
dropped = [c for c in columns_absent_early(train_df, cols_l3, EARLY_CUTOFF)]
cols_l4 = drop_listed(cols_l3, dropped)
funnel.append(("L4", f"Không có dữ liệu trước {EARLY_CUTOFF:%Y-%m}", dropped))

# L5: số gần như hằng số
dropped = [c for c in columns_near_constant(train_df, cols_l4, NEAR_CONSTANT)]
cols_l5 = drop_listed(cols_l4, dropped)
funnel.append(("L5", f"Gần hằng số (>= {NEAR_CONSTANT}% một giá trị)", dropped))

features_from_funnel = cols_l5

In [23]:
# Bảng phễu lọc: số cột bị loại và số cột còn lại sau mỗi bước
remaining = len(cols_start)
rows = []
for step, reason, dropped_cols in funnel:
    remaining -= len(dropped_cols)
    rows.append({"Bước": step, "Quy tắc": reason, "Số cột bị loại": len(dropped_cols), "Số cột còn lại": remaining})
pd.DataFrame(rows).set_index("Bước")

,Quy tắc,Số cột bị loại,Số cột còn lại
Bước,,,
Ban đầu,151 cột của file gốc,0,151
L1,Không biết lúc nộp đơn (theo từ điển),48,103
L2,Văn bản tự do / định danh,4,99
L3,Thiếu trên 60% ở train,34,65
L4,Không có dữ liệu trước 2012-01,34,31
L5,Gần hằng số (>= 98% một giá trị),4,27


In [24]:
# Liệt kê cột bị loại ở mỗi bước từ L2 trở đi (L1 đã nằm trong danh sách ở trên)
for step, reason, dropped_cols in funnel[2:]:
    print(f"{step} - {reason}: {len(dropped_cols)} cột")
    print("   ", ", ".join(dropped_cols) if dropped_cols else "(không có)")

L2 - Văn bản tự do / định danh: 4 cột
    emp_title, desc, title, zip_code
L3 - Thiếu trên 60% ở train: 34 cột
    mths_since_last_record, mths_since_last_major_derog, annual_inc_joint, dti_joint, verification_status_joint, open_acc_6m, open_act_il, open_il_12m, open_il_24m, mths_since_rcnt_il, total_bal_il, il_util, open_rv_12m, open_rv_24m, max_bal_bc, all_util, inq_fi, total_cu_tl, inq_last_12m, mths_since_recent_bc_dlq, mths_since_recent_revol_delinq, revol_bal_joint, sec_app_fico_range_low, sec_app_fico_range_high, sec_app_earliest_cr_line, sec_app_inq_last_6mths, sec_app_mort_acc, sec_app_open_acc, sec_app_revol_util, sec_app_open_act_il, sec_app_num_rev_accts, sec_app_chargeoff_within_12_mths, sec_app_collections_12_mths_ex_med, sec_app_mths_since_last_major_derog
L4 - Không có dữ liệu trước 2012-01: 34 cột
    tot_coll_amt, tot_cur_bal, total_rev_hi_lim, acc_open_past_24mths, avg_cur_bal, bc_open_to_buy, bc_util, mo_sin_old_il_acct, mo_sin_old_rev_tl_op, mo_sin_rcnt_rev_tl_op, 

In [25]:
print(f"Kết quả phễu: {len(cols_start)} -> {len(features_from_funnel)} cột")

Kết quả phễu: 151 -> 27 cột


## 3.2 Whitelist 27 feature và đối chiếu với phễu lọc

Danh sách dưới đây là whitelist chốt, chia theo ba nhóm ý nghĩa. Ô kiểm tra ngay sau đó xác nhận phễu 3.1 cho ra đúng tập cột này.

In [26]:
# Whitelist 27 feature biết được tại thời điểm nộp đơn (Application Time)
FEATURES_27 = [
    # Nhóm 1: Thông tin khoản vay đề nghị (8 cột)
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade", 
    "purpose", "application_type",
    
    # Nhóm 2: Hồ sơ người vay tự khai (6 cột)
    "emp_length", "home_ownership", "annual_inc", "verification_status", 
    "addr_state", "dti",
    
    # Nhóm 3: Lịch sử tín dụng tra cứu từ CIC/Bureau lúc nộp đơn (13 cột)
    "delinq_2yrs", "earliest_cr_line", "fico_range_low", "fico_range_high", 
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util", 
    "total_acc", "pub_rec_bankruptcies", "tax_liens", "mths_since_last_delinq"
]

TARGET_COL = "target"
ALL_28_COLS = FEATURES_27 + [TARGET_COL]

print(f"Tổng số feature lựa chọn: {len(FEATURES_27)} feature")
print(f"Tổng số cột đưa vào xử lý: {len(ALL_28_COLS)} cột (gồm cả biến target)")

Tổng số feature lựa chọn: 27 feature
Tổng số cột đưa vào xử lý: 28 cột (gồm cả biến target)


In [27]:
assert set(features_from_funnel) == set(FEATURES_27), "Lỗi: phễu lọc không trùng whitelist 27 cột!"
print(f"Phễu lọc cho ra đúng {len(features_from_funnel)} cột, trùng khớp whitelist FEATURES_27.")

Phễu lọc cho ra đúng 27 cột, trùng khớp whitelist FEATURES_27.


In [28]:
# Chỉ giữ lại 28 cột hợp lệ
train_28 = train_df[ALL_28_COLS].copy()
val_28 = val_df[ALL_28_COLS].copy()
test_28 = test_df[ALL_28_COLS].copy()

# Giữ thêm cột issue_date cho train/val/test để tính số năm lịch sử tín dụng ở bước sau
train_28["issue_date"] = train_df["issue_date"]
val_28["issue_date"] = val_df["issue_date"]
test_28["issue_date"] = test_df["issue_date"]

In [29]:
print("Kích thước tập Train:", train_28.shape)
print("Kích thước tập Val:  ", val_28.shape)
print("Kích thước tập Test: ", test_28.shape)

assert all(col in train_28.columns for col in ALL_28_COLS), "Lỗi: Thiếu cột trong danh sách 28 cột!"
print(" Xác nhận: 3 tập Train/Val/Test đã được lọc chuẩn theo danh sách 27 feature + 1 target!")

Kích thước tập Train: (941744, 29)
Kích thước tập Val:   (201803, 29)
Kích thước tập Test:  (201803, 29)
 Xác nhận: 3 tập Train/Val/Test đã được lọc chuẩn theo danh sách 27 feature + 1 target!


In [30]:
train_missing_27 = train_28[FEATURES_27].isna().mean() * 100
missing_report = train_missing_27[train_missing_27 > 0].sort_values(ascending=False).to_frame("missing_%")

print(f"Số feature có giá trị thiếu trên tập Train: {len(missing_report)} / 27 feature")
print(missing_report)

Số feature có giá trị thiếu trên tập Train: 7 / 27 feature
                        missing_%
mths_since_last_delinq     50.955
emp_length                  5.369
pub_rec_bankruptcies        0.074
revol_util                  0.054
tax_liens                   0.004
dti                         0.002
inq_last_6mths              0.000


# BƯỚC 4: Tiền xử lý & Xây dựng Scikit-Learn Pipeline chống Data Leakage

In [31]:
def process_features(df):
    df = df.copy()
    
    # 1. Trích xuất số tháng vay từ term (vd: " 36 months" -> 36.0)
    df["term_months"] = df["term"].astype(str).str.extract(r"(\d+)")[0].astype(float)
    
    # 2. Trích xuất số năm công tác từ emp_length (vd: "< 1 year" -> 0, "10+ years" -> 10)
    emp_clean = df["emp_length"].replace({"< 1 year": "0 years"})
    df["emp_length_years"] = emp_clean.astype(str).str.extract(r"(\d+)")[0].astype(float)
    
    # 3. Tính độ dài lịch sử tín dụng (tính theo năm)
    earliest_date = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y")
    df["credit_history_years"] = (df["issue_date"] - earliest_date).dt.days / 365.25
    
    # 4. Gộp FICO cận dưới và cận trên thành điểm trung bình
    df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
    
    # 5. Tỷ lệ số tiền vay trên thu nhập hàng năm (Leverage ratio)
    df["loan_to_income"] = df["loan_amnt"] / (df["annual_inc"] + 1)
    
    return df

In [32]:
train_processed = process_features(train_28)
val_processed = process_features(val_28)
test_processed = process_features(test_28)

print("Đã xử lý xong các đặc trưng số và text cho cả 3 tập!")

Đã xử lý xong các đặc trưng số và text cho cả 3 tập!


In [33]:
# Danh sách biến số (đã thay thế các cột raw bằng các cột đã parse)
NUM_COLS = [
    "loan_amnt", "int_rate", "installment", "annual_inc", "dti",
    "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
    "revol_util", "total_acc", "pub_rec_bankruptcies", "tax_liens",
    "mths_since_last_delinq", "term_months", "emp_length_years",
    "credit_history_years", "fico_avg", "loan_to_income"
]

# Danh sách biến phân loại (Categorical)
CAT_COLS = [
    "grade", "sub_grade", "purpose", "application_type",
    "home_ownership", "verification_status", "addr_state"
]

print(f"Số lượng biến số: {len(NUM_COLS)}")
print(f"Số lượng biến phân loại: {len(CAT_COLS)}")

Số lượng biến số: 20
Số lượng biến phân loại: 7


In [34]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe, NUM_COLS),
    ("cat", categorical_pipe, CAT_COLS)
])

In [35]:
FEATURE_INPUTS = NUM_COLS + CAT_COLS

# Tách X và y
X_train_raw = train_processed[FEATURE_INPUTS]
y_train = train_processed["target"].values

X_val_raw = val_processed[FEATURE_INPUTS]
y_val = val_processed["target"].values

X_test_raw = test_processed[FEATURE_INPUTS]
y_test = test_processed["target"].values

# Fit duy nhất trên Train
preprocessor.fit(X_train_raw)

# Transform sang 3 tập
X_train = preprocessor.transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Kích thước ma trận Train sau mã hóa: {X_train.shape}")
print(f"Kích thước ma trận Val sau mã hóa:   {X_val.shape}")
print(f"Kích thước ma trận Test sau mã hóa:  {X_test.shape}")

Kích thước ma trận Train sau mã hóa: (941744, 138)
Kích thước ma trận Val sau mã hóa:   (201803, 138)
Kích thước ma trận Test sau mã hóa:  (201803, 138)


In [36]:
assert not np.isnan(X_train).any(), "Lỗi: Còn NaN trong Train!"
assert not np.isnan(X_val).any(), "Lỗi: Còn NaN trong Val!"
assert not np.isnan(X_test).any(), "Lỗi: Còn NaN trong Test!"

print(" Xác nhận kiểm tra: Toàn bộ ma trận đã sạch sẽ 100%, sẵn sàng đưa vào mô hình!")

 Xác nhận kiểm tra: Toàn bộ ma trận đã sạch sẽ 100%, sẵn sàng đưa vào mô hình!


# BƯỚC 5: Đánh giá Baseline B1 & Huấn luyện Mô hình Logistic Regression

In [37]:
from sklearn.metrics import roc_auc_score

# Baseline B1: xếp hạng rủi ro của sub_grade (A1 -> G5)
sub_grade_order = sorted(train_processed["sub_grade"].dropna().unique())
sub_grade_map = {grade: i for i, grade in enumerate(sub_grade_order)}

val_subgrade_score = val_processed["sub_grade"].map(sub_grade_map)
auc_baseline = roc_auc_score(y_val, val_subgrade_score)

print(f" ROC-AUC của Baseline B1 (Lending Club sub_grade trên tập Val): {auc_baseline:.4f}")

 ROC-AUC của Baseline B1 (Lending Club sub_grade trên tập Val): 0.6723


In [38]:
from sklearn.linear_model import LogisticRegression

# Huấn luyện Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

print(" Đã hoàn thành huấn luyện Logistic Regression!")

 Đã hoàn thành huấn luyện Logistic Regression!


In [39]:
# Dự báo xác suất vỡ nợ trên tập Val
val_preds = lr_model.predict_proba(X_val)[:, 1]

auc_lr_val = roc_auc_score(y_val, val_preds)
lift_val = auc_lr_val - auc_baseline

print(f" ROC-AUC của Logistic Regression trên tập Val: {auc_lr_val:.4f}")
print(f" Mức cải thiện (Lift) so với Baseline sub_grade: +{lift_val:.4f}")

if lift_val > 0:
    print(" Kết luận: Mô hình của chúng ta thực sự mang lại giá trị cao hơn quy tắc có sẵn của Lending Club!")

 ROC-AUC của Logistic Regression trên tập Val: 0.6934
 Mức cải thiện (Lift) so với Baseline sub_grade: +0.0211
 Kết luận: Mô hình của chúng ta thực sự mang lại giá trị cao hơn quy tắc có sẵn của Lending Club!


In [40]:
# Đánh giá kiểm chứng độc lập cuối cùng trên tập Test (Out-of-Time Test)
test_subgrade_score = test_processed["sub_grade"].map(sub_grade_map)
auc_baseline_test = roc_auc_score(y_test, test_subgrade_score)

test_preds = lr_model.predict_proba(X_test)[:, 1]
auc_lr_test = roc_auc_score(y_test, test_preds)
lift_test = auc_lr_test - auc_baseline_test

print(f" ROC-AUC của Baseline B1 trên tập Test:            {auc_baseline_test:.4f}")
print(f" ROC-AUC của Logistic Regression trên tập Test:     {auc_lr_test:.4f}")
print(f" Mức cải thiện (Lift) trên tập Test:               +{lift_test:.4f}")

 ROC-AUC của Baseline B1 trên tập Test:            0.6825
 ROC-AUC của Logistic Regression trên tập Test:     0.7006
 Mức cải thiện (Lift) trên tập Test:               +0.0181
